# C01 · SAFE P2/P3 · producción final HNC 5%

**Un único notebook, kernel R (IRkernel).** Descarga la fuente oficial, prepara targets y SAFE RANK T2, ejecuta una BO compartida, genera una limpieza OOF de mayo+junio y produce P2 antes que P3. No necesita Drive, scripts auxiliares, caches privados ni HP anteriores. No se envían submissions automáticamente.

## Auditoría previa a los entrenamientos


| Aspecto | Referencias y decisión aplicada |
|---|---|
| Fuente y etiquetas | URL oficial de la consigna. Leads por cliente y período consecutivo de Pasquini. `binaria2=1` para BAJA+1 o BAJA+2; `0` para CONTINUA. Se conservan NA cuando la clase es desconocida; agosto íntegramente sin etiqueta. Conteos contrastados con la tabla de V5.3. |
| Diseño BO | SAFE ejecutado: marzo+abril → junio, FULL, AUC, sin HNC, sin early stopping. Se conserva; no se traslada el split ni el undersampling del ejemplo docente. |
| Presupuesto | El SAFE adjunto tiene **4 LHS + 10 EI**. La consigna pide **10 LHS + 10 EI, máximo 20**, que es lo implementado. El límite de tiempo puede cerrar antes, incluso durante LHS, siempre que exista al menos una evaluación terminada. |
| Espacio | `num_iterations`: entero 64–4096; `num_leaves`: entero 16–2048; Hessian: real 0.000001–0.1. **Los tres son lineales**, sin transformaciones logarítmicas. |
| Optimización | `regr.km`, BFGS, `nugget.estim=TRUE`; EI; `random_search(batch_size=1000)` y terminación de adquisición a 1000 evaluaciones. Son las llamadas del bucle EGO original, separadas en propuesta, evaluación y checkpoint. Semilla 727777, L'Ecuyer-CMRG; reset tras cada evaluación como en el SAFE ejecutado. Ante fallo del surrogate se detiene: no sustituye silenciosamente EI por otra búsqueda. |
| Fijos LightGBM | Configuración SAFE: learning rate 0.005; feature fraction 0.50; bynode 0.20; min data in leaf 0; max bin 31; GBDT binario, AUC, determinista, row-wise, sin undersampling ni early stopping. Lista completa en el bloque 1. `use_missing=TRUE`, `zero_as_missing=FALSE` explícitos como en V5.3/HNC. |
| Ingeniería | `current`, `lag1`, `delta1`, `lag2`, `delta2`; historia por ID y mes calendario real. Corrección de `ccajas_depositos` sólo en mayo, antes de temporales. Signed ranks positivos y negativos separados, con empates promedio, ceros fijos y NA preservados, dentro de cada mes completo. |
| Exclusiones | SAFE antiguo exige KEEP ALL. La consigna prevalece: se retiran de **todos** los modelos las 5 versiones temporales de cada préstamo y sus ranks. El crudo queda intacto. Se conservan las demás exclusiones SAFE de fechas, metadatos y variables no numéricas. El esquema de las referencias tiene 144 fuentes/720 ranks; al retirar dos fuentes se esperan 710 predictores, pero el manifiesto se deriva de la fuente, sin imponer 740. |
| HNC | Una semilla 727777, cinco folds agrupados por cliente y estratificados por `max(binaria2)` del cliente, sobre mayo+junio. Cada mes retira `floor(n_CONTINUA*5/100)`, por OOF descendente y **ID convertido a texto** en los empates, como HNC V1. Conserva todas las BAJA. No hay screening de porcentajes ni HP transferidos. |
| RNG de HNC | El HNC adjunto usa `set.seed(seed+y)` sin fijar `RNGkind`. Se explicita Mersenne-Twister/Inversion/Rejection, default de una sesión R autónoma, para que no herede L'Ecuyer de la BO. Esta elección queda registrada en la firma HNC. |
| Producción | P2: mayo+junio limpios → agosto. P3: marzo+abril completos + la misma limpieza de mayo+junio → agosto. Ambos usan los mismos HP y features. Los ranks no se recalculan después de excluir filas. |
| Semillas | Se conservan exactamente las 10 semillas y el orden de `PARAM$seeds_final`. **No hay una secuencia histórica 11–50 en los archivos.** Para satisfacer la ampliación se incorpora una extensión determinista explícita, independiente del RNG de BO/OOF, congelada en `config/base/semillas.csv`; no se presenta como una secuencia de la referencia. |
| Cutoffs | Se toman del SAFE principal: 9500, 10000, 10500, 11000, 11500, 12000, 12500, 13000, 13500. HNC V1 usa otra grilla (11000–15000); aquí prevalece la grilla SAFE que pide la consigna. Una columna de IDs, **sin encabezado**. |

**Ganancia de junio:** +1.072.500 por BAJA+2 y −27.500 por las demás observaciones, como las referencias. Se guarda la curva para **todos** los cutoffs y BAJA+2 acumuladas. GainBand es la media de ganancias en 11.500, 12.000, 12.500, 13.000 y 13.500, conforme al SAFE original. HP seleccionado exclusivamente por AUC; empates por evaluación más temprana. La ganancia es evidencia de la **validación temporal sin HNC** utilizada para seleccionar HP, no una estimación independiente de P2/P3 limpios. No se calcula una supuesta ganancia real de agosto sin etiquetas.

Con los conteos oficiales, HNC debe retirar **8.087 CONTINUA de mayo y 8.107 de junio**. Se esperan 311.688 filas de train P2 y 637.872 de P3; la identidad de las filas excluidas depende de las OOF generadas con los HP ganadores.

## Checkpoints y recuperación

- Cada evaluación BO conserva predicciones, curva, métricas, historial, diseño LHS, archivo de observaciones de `bbotk`, surrogate y adquisición de `mlr3mbo`, RNG y tiempo acumulado. **La propuesta también se guarda antes de entrenar.** Una interrupción en la 17 conserva las 16 confirmadas y permite retomar exactamente la propuesta pendiente.
- Cada fold y semilla guarda predicciones, HP, firma de features y metadatos. La limpieza guarda todas las filas OOF y su exclusión, el fold, los conteos y la comprobación de BAJA conservadas.
- Los archivos se escriben atómicamente. En GCS se suben como objetos identificados por MD5 y se comprueban tamaño y MD5 remoto; un commit se publica **al final**. Un error de subida detiene la ejecución antes del próximo trabajo. Un resultado local terminado cuyo respaldo falló se respalda al reanudar sin volver a entrenarlo.
- En una VM nueva, el notebook descarga automáticamente los commits y archivos que necesita. Los caches de ranks son locales y regenerables desde el crudo; los resultados costosos, el estado BO y los manifiestos están en GCS. No se borra ni sincroniza destructivamente el bucket.
- Las firmas incluyen parámetros, datos, orden de features, limpieza y versiones de R/paquetes. Un conflicto se detiene; no se reemplazan resultados incompatibles. Para retomar se necesitan las mismas versiones registradas en `config/base/config.rds` y `sessionInfo.txt`. Un cambio metodológico o de versiones requiere otro experimento/prefijo, o restablecer el entorno anterior.
- Una vez elegidos los HP, `bo/seleccion` queda congelado. Reanudar o ampliar semillas no continúa una BO cerrada ni cambia silenciosamente la limpieza. Antes de cerrar por tiempo se puede ampliar `bo_horas`; después de la selección, una BO distinta requiere otro experimento.

Ejecutar **una sola instancia activa** por carpeta/prefijo. No reutilizar este nombre para otro diseño.

## Uso

1. **Desde cero:** abrir con kernel R, revisar el bloque 1 y ejecutar todas las celdas en orden. Instala sólo paquetes faltantes. La VM necesita `gcloud` autenticado con lectura/escritura en el prefijo configurado. No modifica credenciales ni instala gcloud.
2. **Reanudar:** en esta VM o una nueva, conservar `EXPERIMENTO`, `GCS`, parámetros metodológicos y versiones; volver a ejecutar desde arriba. La raíz local puede cambiar.
3. **Tiempo:** `bo_horas=4` es el presupuesto activo acumulado por defecto; no limita un modelo ya iniciado. `limite_ART` y `margen_minutos=30` impiden iniciar otro trabajo pasada la reserva para el cierre. Tampoco cortan un modelo en curso: un entrenamiento largo puede sobrepasar ese horario. Para reproducción posterior al 11/10/2026, usar `limite_ART <- NA_character_`.
4. **Familias:** cambiar `producir_P2` y `producir_P3`. Si P2 está activo, P3 sólo empieza cuando P2 alcanza el número solicitado y su ensemble/submissions están confirmados. Si el tiempo deja P2 parcial, se conserva y P3 espera. Para producir sólo P3, desactivar P2.
5. **10 → 20 → 50:** cambiar únicamente `n_semillas`. No cambiar la secuencia, HP, features ni HNC. Sólo entrena semillas faltantes; los ensembles anteriores permanecen identificados por el número y conjunto exacto de semillas.
6. **Parciales:** se exporta y respalda un ensemble después de cada semilla terminada. `solo_ensembles <- TRUE` recupera/exporta lo ya disponible sin entrenar; requiere que BO y HNC estén completadas. Los metadatos registran el número real, las semillas y el objetivo al crear el archivo.
7. **GCS:** `respaldo_GCS=TRUE` es obligatorio para esta producción. `SAFE_FINAL_GCS` o `GCS` permiten otro prefijo. Un evaluador externo puede poner `respaldo_GCS <- FALSE`: el pipeline sigue siendo reproducible desde la URL pública, pero los resultados quedarán sólo en su almacenamiento local.

Las salidas están en `OUT/checkpoints`: `bo`, `hnc`, `produccion/P2`, `ensembles/P2`, luego `produccion/P3`, `ensembles/P3`. Cada ensemble contiene probabilidades, ranking completo, nueve submissions y su manifiesto. Los nombres identifican familia, evaluación HP, firma, versión HNC, número de semillas y cutoff. No se mezclan P2 y P3 ni se incluyen metaensembles con E012. Los modelos binarios/texto de LightGBM son opcionales (`guardar_modelos`); predicciones y metadatos siempre se guardan.

## Costo orientativo y validación del entregable

El máximo inicial es **45 entrenamientos: 20 BO + 5 OOF + 10 P2 + 10 P3**. En las salidas adjuntas la BO original acumuló 194,4 minutos para 14 evaluaciones (0,6–23,8 min cada una); P2 rondó 17,2–18,4 min/semilla y P3 30,1–35,4. Son mediciones de otros HP y entorno, no promesas para esta corrida.

Con 10 semillas, producción sola ≈8–9 h; sumar BO, cinco OOF, preparación, escrituras y GCS. Una previsión práctica de **12–16 h** tiene incertidumbre; recalibrar con los tiempos guardados de las primeras evaluaciones. Veinte semillas por familia pueden llevar el total a ≈20–25 h; cincuenta probablemente excedan el cierre. No se estima dinero sin la tarifa de la VM. P2 tiene prioridad y los presupuestos evitan exigir 20 evaluaciones para obtener una producción útil.

La auditoría realizada y sus límites se registran al final del notebook. Ningún entrenamiento sobre el dataset de la competencia se ejecutó durante la construcción del entregable.

Referencias técnicas de implementación: [bucle EGO de mlr3mbo](https://mlr3mbo.mlr-org.com/reference/mlr_loop_functions_ego.html), [metadatos de objetos GCS](https://docs.cloud.google.com/sdk/gcloud/reference/storage/objects/describe), [precondiciones de escritura GCS](https://docs.cloud.google.com/storage/docs/request-preconditions). La metodología y los parámetros proceden de los notebooks adjuntos y la consigna.


In [1]:
# BLOQUE 1 — Configuración y bibliotecas #####
options(repos=c(CRAN="https://cloud.r-project.org"), warn=1, timeout=1800)

ROOT <- Sys.getenv("SAFE_ROOT", unset=if (dir.exists("/home/ds"))
  "/home/ds/Expe" else file.path(path.expand("~"), "Expe"))
EXPERIMENTO <- "SAFE_P2_P3_FINAL_HNC5_V1"
OUT <- file.path(ROOT, "experimentos", EXPERIMENTO)
crudo_oficial <- file.path(OUT, "fuentes", "competencia_01_crudo.csv")
url_oficial <- "https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/competencia_01_crudo.csv"

producir_P2 <- TRUE
producir_P3 <- FALSE
n_semillas <- 10L                       # 10, 20 o 50; conserva las ya terminadas
bo_horas <- 8                           # presupuesto activo ACUMULADO; máximo 20 evaluaciones
solo_ensembles <- FALSE                 # TRUE: exporta lo disponible, sin nuevos entrenamientos
guardar_modelos <- FALSE                # opcional; predicciones y metadatos se guardan siempre

# En esta VM es obligatorio. Para reproducir fuera del entorno: respaldo_GCS <- FALSE.
respaldo_GCS <- FALSE
GCS <- Sys.getenv("SAFE_FINAL_GCS", unset=paste0(
  "gs://pabloparisidat_buckito2026/Expe/experimentos/", EXPERIMENTO))

# No se inicia un trabajo después de este límite. No interrumpe un modelo a mitad.
# Para reproducir después de la competencia: limite_ART <- NA_character_.
limite_ART <- NA_character_
margen_minutos <- 30                    # reserva para terminar / exportar / subir submission
limite <- if (is.na(limite_ART)) as.POSIXct(NA) else
  as.POSIXct(limite_ART, tz="America/Argentina/Buenos_Aires") - margen_minutos*60

stopifnot(n_semillas %in% c(10L,20L,50L), length(bo_horas)==1L,
          !is.na(bo_horas), bo_horas >= 0, margen_minutos >= 0)
dir.create(OUT, recursive=TRUE, showWarnings=FALSE)
dir.create(file.path(ROOT,"R_libs"), recursive=TRUE, showWarnings=FALSE)
.libPaths(c(file.path(ROOT,"R_libs"), .libPaths()))
paquetes <- c("data.table","lightgbm","DiceKriging","mlr3","mlr3mbo",
              "mlr3learners","paradox","bbotk","lhs","jsonlite","rgenoud")
faltan <- paquetes[!vapply(paquetes, requireNamespace, logical(1), quietly=TRUE)]
if (length(faltan)) install.packages(faltan, lib=.libPaths()[1], Ncpus=2L)
faltan <- paquetes[!vapply(paquetes, requireNamespace, logical(1), quietly=TRUE)]
if (length(faltan)) stop("Faltan paquetes: ", paste(faltan, collapse=", "))
suppressPackageStartupMessages({
  library(data.table); library(lightgbm); library(DiceKriging)
  library(mlr3); library(mlr3mbo); library(mlr3learners)
  library(paradox); library(bbotk)
})

# Exactamente las diez semillas presentes en las referencias, en su orden original.
semillas_originales <- c(727777L,773777L,775777L,777737L,787777L,
                        777727L,777773L,777775L,777787L,797777L)
# Las referencias NO especifican semillas 11–50. Extensión operativa explícita,
# independiente del RNG de BO/OOF; se congela íntegra desde la primera ejecución.
set.seed(727777L, kind="L'Ecuyer-CMRG", normal.kind="Inversion", sample.kind="Rejection")
semillas_50 <- c(semillas_originales,
  head(setdiff(sample.int(1000000000L, 100L), semillas_originales), 40L))
stopifnot(length(semillas_50)==50L, !anyDuplicated(semillas_50))

PARAM <- list(
  version="SAFE_P2_P3_FINAL_HNC5_V1_20261011",
  BO_train=c(202103L,202104L), BO_valid=202106L,
  meses_cache=c(202103L,202104L,202105L,202106L,202108L),
  P_final=list(P2=c(202105L,202106L), P3=c(202103L,202104L,202105L,202106L)),
  mes_score=202108L, seed_BO=727777L, seed_OOF=727777L,
  n_iniciales=10L, n_EI=10L, n_total=20L, n_folds=5L, hnc_pct=5,
  excluidas=c("cprestamos_personales","mprestamos_personales"),
  cortes=seq(9500L,15000L,500L), gainband=seq(11500L,15000L,500L),
  ganancia_baja2=1072500, costo_estimulo=27500,
  semillas=semillas_50,
  space=list(num_iterations=c(64L,4096L), num_leaves=c(16L,2048L),
             min_sum_hessian_in_leaf=c(1e-6,0.1)),
  metadata=c("numero_de_cliente","foto_mes","clase_ternaria","clase01",
             "binaria2","target","azar","training","fold"),
  fechas="fecha|fvencimiento|finiciomora|fechaalta|fultimo|ultimo_cierre"
)
PARAM$lgbm_fijos <- list(
  boosting="gbdt", objective="binary", metric="auc", feature_pre_filter=FALSE,
  first_metric_only=FALSE, boost_from_average=TRUE, force_row_wise=TRUE,
  use_missing=TRUE, zero_as_missing=FALSE, deterministic=TRUE, verbosity=-100,
  max_depth=-1L, min_gain_to_split=0, lambda_l1=0.0, lambda_l2=0.0,
  bagging_fraction=1.0, pos_bagging_fraction=1.0, neg_bagging_fraction=1.0,
  is_unbalance=FALSE, scale_pos_weight=1.0, drop_rate=0.1, max_drop=50,
  skip_drop=0.5, extra_trees=FALSE, early_stopping=0,
  learning_rate=0.005, feature_fraction=0.50, feature_fraction_bynode=0.20,
  min_data_in_leaf=0L, max_bin=31L
)
versiones <- c(R=as.character(getRversion()),
  vapply(paquetes, function(p) as.character(packageVersion(p)), character(1)))
# Lista estable: no depende de qué celdas se ejecutaron antes en este kernel.
dependencias <- sort(unique(c(paquetes,unlist(tools::package_dependencies(
  paquetes,db=installed.packages(),which=c("Depends","Imports","LinkingTo"),recursive=TRUE)))))
dependencias <- setdiff(dependencias,"R")
versiones_dependencias <- vapply(dependencias,
  function(p) as.character(packageVersion(p)), character(1))
hay_tiempo <- function() is.na(limite) || Sys.time() < limite
message("Salida: ", OUT, " | GCS obligatorio: ", respaldo_GCS,
        " | P2=", producir_P2, " | P3=", producir_P3, " | semillas=", n_semillas)


Salida: /home/ds/Expe/experimentos/SAFE_P2_P3_FINAL_HNC5_V1 | GCS obligatorio: TRUE | P2=TRUE | P3=TRUE | semillas=10



In [2]:
# BLOQUE 2 — Escritura atómica y checkpoints confirmados en GCS #####
md5 <- function(f) unname(tools::md5sum(f))
firma <- function(x) {
  f <- tempfile(); on.exit(unlink(f), add=TRUE)
  saveRDS(x, f, compress=FALSE, version=3)
  md5(f)
}
rds_atomico <- function(x, f) {
  dir.create(dirname(f), recursive=TRUE, showWarnings=FALSE)
  t <- paste0(f,".tmp_",Sys.getpid()); on.exit(unlink(t), add=TRUE)
  saveRDS(x, t, compress="gzip", version=3)
  invisible(readRDS(t))
  if (!file.rename(t,f)) stop("No se pudo promover: ", f)
  invisible(f)
}
csv_atomico <- function(x, f, encabezado=TRUE) {
  dir.create(dirname(f), recursive=TRUE, showWarnings=FALSE)
  t <- paste0(f,".tmp_",Sys.getpid()); on.exit(unlink(t), add=TRUE)
  fwrite(x, t, col.names=encabezado, na="NA")
  z <- fread(t, header=encabezado, na.strings=c("","NA"), showProgress=FALSE)
  if (!identical(dim(z),dim(x))) stop("CSV cambió dimensiones: ", f)
  if (encabezado && !identical(names(z),names(x))) stop("CSV cambió columnas: ",f)
  for (j in seq_along(x)) {
    if (!identical(is.na(z[[j]]),is.na(x[[j]]))) stop("CSV cambió NA: ",f)
    ok <- if (is.numeric(x[[j]])) isTRUE(all.equal(as.numeric(z[[j]]),
      as.numeric(x[[j]]), tolerance=1e-12, check.attributes=FALSE)) else
      identical(as.character(z[[j]]),as.character(x[[j]]))
    if (!ok) stop("CSV cambió valores: ",f)
  }
  if (!file.rename(t,f)) stop("No se pudo promover CSV: ",f)
  invisible(f)
}
CONFIG_ID <- firma(list(PARAM=PARAM, versiones=versiones,
                       versiones_dependencias=versiones_dependencias))
GCS <- sub("/+$", "", GCS)
gcloud <- unname(Sys.which("gcloud"))
if (respaldo_GCS && (!nzchar(gcloud) || !grepl("^gs://[^/]+/.+",GCS)))
  stop("Configurar gcloud y GCS con un prefijo propio, o desactivar GCS fuera de esta VM.")

gcs_comando <- function(args, permitir_ausente=FALSE) {
  err <- tempfile(); on.exit(unlink(err), add=TRUE)
  out <- suppressWarnings(system2(gcloud, vapply(c(args,"--quiet"),shQuote,character(1)),
                                  stdout=TRUE, stderr=err))
  status <- attr(out,"status")
  if (!is.null(status) && status!=0L) {
    detalle <- paste(readLines(err,warn=FALSE),collapse="\n")
    # Sólo una ausencia inequívoca es un experimento/objeto nuevo. 403/red NO lo son.
    if (permitir_ausente && grepl("(?i)(404|not found|no such object|matched no objects|no urls matched|does not exist)",
                                 detalle,perl=TRUE) &&
        !grepl("(?i)(403|permission|forbidden|access denied)",detalle,perl=TRUE)) return(NULL)
    stop("GCS falló; no se inicia el siguiente trabajo. ", detalle)
  }
  out
}
gcs_info <- function(uri) {
  z <- gcs_comando(c("storage","objects","describe",uri,"--raw","--format=json"),TRUE)
  if (is.null(z)) return(NULL)
  jsonlite::fromJSON(paste(z,collapse="\n"))
}
md5_base64 <- function(h) jsonlite::base64_enc(as.raw(strtoi(
  substring(h,seq(1,31,2),seq(2,32,2)),16L)))
verificar_remoto <- function(info, f) {
  !is.null(info) && identical(info$md5Hash,md5_base64(md5(f))) &&
    isTRUE(as.numeric(info$size)==file.info(f)$size)
}
subir_inmutable <- function(f, uri) {
  info <- gcs_info(uri)
  if (is.null(info)) {
    gcs_comando(c("storage","cp",f,uri,"--if-generation-match=0",
                  paste0("--content-md5=",md5_base64(md5(f)))))
    info <- gcs_info(uri)
  }
  if (!verificar_remoto(info,f)) stop("Objeto remoto diferente o sin MD5 verificable: ",uri)
  invisible(TRUE)
}
descargar_verificado <- function(uri, f, hash=NULL) {
  info <- gcs_info(uri)
  if (is.null(info)) stop("Falta un objeto de un checkpoint confirmado: ",uri)
  dir.create(dirname(f),recursive=TRUE,showWarnings=FALSE)
  t <- paste0(f,".download_",Sys.getpid()); on.exit(unlink(t),add=TRUE)
  gcs_comando(c("storage","cp",uri,t))
  if (!verificar_remoto(info,t) || (!is.null(hash) && md5(t)!=hash))
    stop("Descarga no verificable: ",uri)
  if (file.exists(f) && md5(f)!=md5(t)) stop("Conflicto local, no se sobrescribe: ",f)
  if (!file.exists(f) && !file.rename(t,f)) stop("No se pudo recuperar: ",f)
  invisible(f)
}
ck_dir <- function(key) file.path(OUT,"checkpoints",key)
ck_uri <- function(key) paste0(GCS,"/checkpoints/",key,".rds")
confirmados_sesion <- new.env(parent=emptyenv())
clave_confirmacion <- function(key) paste(GCS,CONFIG_ID,key,sep="|")

validar_commit <- function(z,key,sig=NULL) {
  if (!is.list(z) || !identical(z$key,key) || !identical(z$config_id,CONFIG_ID) ||
      (!is.null(sig) && !identical(z$firma,sig)))
    stop("Checkpoint de otro experimento/configuración: ",key)
  a <- z$archivos
  if (!nrow(a) || anyDuplicated(a$archivo) || anyNA(a) ||
      any(basename(a$archivo)!=a$archivo) || any(!grepl("^[a-f0-9]{32}$",a$md5)))
    stop("Manifiesto de archivos inválido: ",key)
  invisible(TRUE)
}
respaldar <- function(key,z) {
  if (!respaldo_GCS) return(invisible(TRUE))
  for (i in seq_len(nrow(z$archivos))) {
    a <- z$archivos[i,]
    subir_inmutable(file.path(ck_dir(key),a$archivo),paste0(GCS,"/objetos/",a$md5,"/",a$archivo))
  }
  # El commit remoto se publica ÚLTIMO. Sólo él declara terminado el trabajo.
  subir_inmutable(file.path(ck_dir(key),"COMMIT.rds"),ck_uri(key))
  assign(clave_confirmacion(key),md5(file.path(ck_dir(key),"COMMIT.rds")),
         envir=confirmados_sesion)
  invisible(TRUE)
}
abrir_checkpoint <- function(key,sig=NULL,remoto=TRUE) {
  d <- ck_dir(key); f <- file.path(d,"COMMIT.rds")
  confirmado <- file.exists(f) && exists(clave_confirmacion(key),envir=confirmados_sesion,inherits=FALSE) &&
    identical(get(clave_confirmacion(key),envir=confirmados_sesion),md5(f))
  info <- if (respaldo_GCS && remoto && !confirmado) gcs_info(ck_uri(key)) else NULL
  if (!file.exists(f) && !is.null(info)) {
    descargar_verificado(ck_uri(key),f)
    message("Recuperado desde GCS: ",key)
  }
  if (!file.exists(f)) return(NULL)
  z <- readRDS(f); validar_commit(z,key,sig)
  if (!is.null(info) && !verificar_remoto(info,f)) stop("Conflicto de commits: ",key)
  for (i in seq_len(nrow(z$archivos))) {
    a <- z$archivos[i,]; p <- file.path(d,a$archivo)
    if (!file.exists(p) && respaldo_GCS && remoto)
      descargar_verificado(paste0(GCS,"/objetos/",a$md5,"/",a$archivo),p,a$md5)
    if (!file.exists(p) || md5(p)!=a$md5 || file.info(p)$size!=a$bytes)
      stop("Checkpoint corrupto/incompleto: ",key," / ",a$archivo)
  }
  # Si el backup falló luego del commit local, ahora se completa sin reentrenar.
  if (remoto && !confirmado) respaldar(key,z)
  z
}
iniciar_checkpoint <- function(key) {
  d <- ck_dir(key)
  if (file.exists(file.path(d,"COMMIT.rds"))) stop("No sobrescribir checkpoint: ",key)
  if (dir.exists(d) && length(list.files(d,all.files=TRUE,no..=TRUE))) {
    destino <- paste0(d,"_incompleto_",format(Sys.time(),"%Y%m%d%H%M%OS6"),"_",Sys.getpid())
    if (!file.rename(d,destino)) stop("No se pudo apartar intento incompleto: ",d)
    message("Intento interrumpido conservado: ",destino)
  }
  dir.create(d,recursive=TRUE,showWarnings=FALSE)
  d
}
cerrar_checkpoint <- function(key,sig,remoto=TRUE) {
  d <- ck_dir(key)
  fs <- list.files(d,full.names=TRUE)
  fs <- fs[!grepl("COMMIT\\.rds$|\\.tmp_|\\.download_",fs)]
  stopifnot(length(fs)>0L, !any(file.info(fs)$isdir))
  z <- list(key=key, config_id=CONFIG_ID, firma=sig,
    archivos=data.frame(archivo=basename(fs),md5=unname(tools::md5sum(fs)),
                         bytes=unname(file.info(fs)$size),stringsAsFactors=FALSE))
  validar_commit(z,key,sig)
  rds_atomico(z,file.path(d,"COMMIT.rds"))
  if (remoto) respaldar(key,z)             # ERROR de GCS se propaga; nunca un mero warning
  invisible(z)
}
if (respaldo_GCS) {
  bucket <- sub("^(gs://[^/]+).*", "\\1",GCS)
  invisible(gcs_comando(c("storage","buckets","describe",bucket,"--format=json")))
}
if (is.null(abrir_checkpoint("config/base",CONFIG_ID))) {
  d <- iniciar_checkpoint("config/base")
  rds_atomico(list(PARAM=PARAM,versiones=versiones,dependencias=versiones_dependencias,
                   config_id=CONFIG_ID),file.path(d,"config.rds"))
  writeLines(capture.output(sessionInfo()),file.path(d,"sessionInfo.txt"))
  csv_atomico(data.table(orden=seq_along(semillas_50),seed=semillas_50,
    origen=c(rep("SAFE_original",10),rep("extension_determinista",40))),file.path(d,"semillas.csv"))
  cerrar_checkpoint("config/base",CONFIG_ID)
}


In [9]:
# BLOQUE 3 — Targets, ranks, métricas y entrenamiento SAFE #####
generar_targets <- function(dataset) {
  if (anyNA(dataset$numero_de_cliente) || anyNA(dataset$foto_mes) ||
      anyDuplicated(dataset,by=c("numero_de_cliente","foto_mes")))
    stop("Crudo: IDs/meses faltantes o duplicados cliente-mes.")
  dsimple <- dataset[,.(pos=.I,numero_de_cliente,
    periodo0=as.integer(foto_mes/100)*12L + foto_mes%%100L)]
  setorder(dsimple,numero_de_cliente,periodo0)
  periodo_ultimo <- max(dsimple$periodo0)
  periodo_anteultimo <- periodo_ultimo - 1L
  dsimple[,c("periodo1","periodo2") := shift(periodo0,n=1:2,fill=NA,type="lead"),
          by=numero_de_cliente]
  dsimple[,clase_ternaria := NA_character_]
  dsimple[periodo0 < periodo_anteultimo,clase_ternaria := "CONTINUA"]
  dsimple[periodo0 < periodo_ultimo & (is.na(periodo1) | periodo0+1L < periodo1),
          clase_ternaria := "BAJA+1"]
  dsimple[periodo0 < periodo_anteultimo & periodo0+1L==periodo1 &
          (is.na(periodo2) | periodo0+2L < periodo2),clase_ternaria := "BAJA+2"]
  setorder(dsimple,pos)
  dataset[,clase_ternaria := dsimple$clase_ternaria]
  dataset[,binaria2 := fifelse(is.na(clase_ternaria),NA_integer_,
    as.integer(clase_ternaria %in% c("BAJA+1","BAJA+2")))]
  dataset
}
signed_rank <- function(x) {
  z <- rep(NA_real_,length(x))
  neg <- !is.na(x) & x < 0
  cer <- !is.na(x) & x == 0
  pos <- !is.na(x) & x > 0
  if (any(neg)) z[neg] <- -frank(abs(x[neg]),ties.method="average")/sum(neg)
  z[cer] <- 0
  if (any(pos)) z[pos] <- frank(x[pos],ties.method="average")/sum(pos)
  z
}
mes_anterior <- function(ym,k) {
  total <- (ym%/%100L)*12L + ym%%100L - 1L - as.integer(k)
  as.integer((total%/%12L)*100L + total%%12L + 1L)
}
validar_prob <- function(p,n=length(p)) {
  if (length(p)!=n || anyNA(p) || any(!is.finite(p)) || any(p<0 | p>1))
    stop("Predicciones incompletas/no finitas/fuera de [0,1].")
  invisible(TRUE)
}
validar_ids <- function(ids,referencia) {
  if (anyNA(ids) || anyDuplicated(ids) ||
      !identical(as.character(ids),as.character(referencia)))
    stop("IDs/orden diferentes del universo de referencia.")
  invisible(TRUE)
}
validar_target <- function(d,etiquetado=TRUE) {
  if (!identical(is.na(d$clase_ternaria),is.na(d$binaria2)) ||
      any(!is.na(d$clase_ternaria) & !d$clase_ternaria %in% c("CONTINUA","BAJA+1","BAJA+2")))
    stop("Target desconocido o máscara NA alterada.")
  if (etiquetado && (anyNA(d$binaria2) ||
      any(d$binaria2!=as.integer(d$clase_ternaria %in% c("BAJA+1","BAJA+2")))))
    stop("Target binaria2 incorrecto.")
  if (!etiquetado && (!all(is.na(d$clase_ternaria)) || !all(is.na(d$binaria2))))
    stop("Agosto debe carecer de etiquetas y target.")
  invisible(TRUE)
}
auc_binaria <- function(y,p) {
  validar_prob(p,length(y))
  if (anyNA(y) || any(!y %in% 0:1)) stop("AUC: target inválido.")
  n1 <- sum(y==1L); n0 <- sum(y==0L)
  if (!n1 || !n0) stop("AUC requiere las dos clases.")
  (sum(rank(p,ties.method="average")[y==1L])-n1*(n1+1)/2)/(n1*n0)
}
curva_ganancia <- function(d,p) {
  validar_prob(p,nrow(d))
  x <- d[,.(numero_de_cliente,clase_ternaria)]
  x[,prediction := p]
  setorder(x,-prediction,numero_de_cliente)
  x[,baja2 := as.integer(clase_ternaria=="BAJA+2")]
  x[,.(cutoff=seq_len(.N),numero_de_cliente,prediction,
    bajas2_acumuladas=cumsum(baja2),
    ganancia_acumulada=cumsum(fifelse(baja2==1L,PARAM$ganancia_baja2,-PARAM$costo_estimulo)))]
}
hp_campos <- c("num_iterations","num_leaves","min_sum_hessian_in_leaf")
hp_lista <- function(hp) list(num_iterations=as.integer(hp$num_iterations),
  num_leaves=as.integer(hp$num_leaves),
  min_sum_hessian_in_leaf=as.numeric(hp$min_sum_hessian_in_leaf))
param_modelo <- function(hp,seed) modifyList(PARAM$lgbm_fijos,
  c(hp_lista(hp),list(seed=as.integer(seed))))
matriz_SAFE <- function(d) {
  x <- data.matrix(d[,..FEATURES])
  stopifnot(identical(colnames(x),FEATURES),nrow(x)==nrow(d))
  for (j in seq_along(FEATURES)) {
    if (!identical(is.na(x[,j]),is.na(d[[FEATURES[j]]])) ||
        any(!is.na(x[,j]) & !is.finite(x[,j]))) stop("Matriz alteró NA/valores: ",FEATURES[j])
  }
  x
}
# Todas las etapas llaman a esta misma función con el MISMO manifiesto y parámetros fijos.
ajustar_SAFE <- function(X,y,Xscore,hp,seed,valid_y=NULL,modelo_archivo=NULL) {
  stopifnot(identical(colnames(X),FEATURES),identical(colnames(Xscore),FEATURES),
            ncol(X)==length(FEATURES),!anyNA(y),all(y %in% 0:1))
  t0 <- proc.time()[["elapsed"]]
  dtrain <- lgb.Dataset(X,label=y,params=list(max_bin=31L,feature_pre_filter=FALSE,
                                            use_missing=TRUE,zero_as_missing=FALSE))
  #valids <- if (is.null(valid_y)) list() else list(val=lgb.Dataset(Xscore,label=valid_y,
   # params=list(max_bin=31L,feature_pre_filter=FALSE,use_missing=TRUE,zero_as_missing=FALSE)))
  model <- lgb.train(params=param_modelo(hp,seed),data=dtrain)
  t1 <- proc.time()[["elapsed"]]
  p <- as.numeric(predict(model,Xscore))
  validar_prob(p,nrow(Xscore))
  if (!is.null(modelo_archivo)) {
    t <- paste0(modelo_archivo,".tmp_",Sys.getpid())
    on.exit(unlink(t),add=TRUE)
    lgb.save(model,t)
    invisible(lgb.load(t))
    if (!file.rename(t,modelo_archivo)) stop("No se pudo guardar modelo.")
  }
  list(p=p,seg_entrenamiento=t1-t0,seg_prediccion=proc.time()[["elapsed"]]-t1)
}


In [4]:
# BLOQUE 3b — Comprobaciones pequeñas, sin entrenamientos #####
local({
  x <- c(NA_real_,-4,-1,-1,0,1,2,2)
  esperado <- c(NA_real_,-1,-0.5,-0.5,0,1/3,5/6,5/6)
  stopifnot(isTRUE(all.equal(signed_rank(x),esperado)),
            identical(is.na(signed_rank(x)),is.na(x)),
            mes_anterior(202101L,1L)==202012L,
            mes_anterior(202103L,2L)==202101L)
  ejemplo <- data.table(numero_de_cliente=c(rep(1L,6),rep(2L,3),rep(3L,4)),
    foto_mes=c(202103L:202108L,202103L:202105L,202103L,202105L,202106L,202108L))
  ejemplo <- generar_targets(ejemplo)
  stopifnot(ejemplo[numero_de_cliente==2L & foto_mes==202104L,clase_ternaria]=="BAJA+2",
            ejemplo[numero_de_cliente==2L & foto_mes==202105L,clase_ternaria]=="BAJA+1",
            ejemplo[numero_de_cliente==3L & foto_mes==202103L,clase_ternaria]=="BAJA+1",
            ejemplo[numero_de_cliente==1L & foto_mes==202106L,binaria2]==0L,
            all(is.na(ejemplo[foto_mes==202108L,binaria2])))
  z <- data.table(numero=c(NA_real_,0,0.125),texto=c(NA_character_,"a","b"))
  f <- tempfile(fileext=".csv"); on.exit(unlink(f),add=TRUE)
  csv_atomico(z,f)
  stopifnot(identical(is.na(fread(f,na.strings=c("","NA"))),is.na(z)))
  message("Pruebas pequeñas OK: targets, huecos de historia, signos, ceros y NA.")
})


Pruebas pequeñas OK: targets, huecos de historia, signos, ceros y NA.



In [5]:
# BLOQUE 4 — Fuente oficial y targets de Pasquini #####
dir.create(dirname(crudo_oficial),recursive=TRUE,showWarnings=FALSE)
if (!file.exists(crudo_oficial)) {
  t <- paste0(crudo_oficial,".descarga")
  if (file.exists(t)) unlink(t)           # sólo la descarga incompleta de esta salida
  download.file(url_oficial,t,mode="wb",quiet=FALSE)
  if (!file.exists(t) || file.info(t)$size==0) stop("Descarga vacía.")
  if (!file.rename(t,crudo_oficial)) stop("No se pudo promover la descarga.")
}
raw_md5 <- md5(crudo_oficial)
dataset <- fread(crudo_oficial,na.strings=c("","NA"),showProgress=TRUE)
requeridas <- c("numero_de_cliente","foto_mes","ccajas_depositos",PARAM$excluidas)
if (length(setdiff(requeridas,names(dataset)))) stop("Fuente oficial: faltan columnas.")
if (!identical(sort(unique(dataset$foto_mes)),202103L:202108L))
  stop("Meses de la fuente distintos de marzo–agosto 2021.")

# Mismo criterio y orden de columnas que SAFE; muestra de 5.000 filas, explícitamente NA.
muestra <- fread(crudo_oficial,nrows=5000L,na.strings=c("","NA"),stringsAsFactors=TRUE)
fuentes <- data.table(variable=names(muestra),tipo=vapply(muestra,
  function(x) paste(class(x),collapse="/"),character(1)))
fuentes[,decision := vapply(variable,function(v) {
  if (v %in% PARAM$metadata || grepl("^clase",tolower(v))) return("METADATA_TARGET")
  if (grepl(PARAM$fechas,tolower(v),perl=TRUE)) return("FECHA_METADATA")
  if (is.numeric(muestra[[v]]) || is.integer(muestra[[v]]) || is.logical(muestra[[v]]))
    return("RANK_PREDICTOR")
  "NO_NUMERICA"
},character(1))]
CURRENT <- fuentes[decision=="RANK_PREDICTOR",variable]
stopifnot(length(CURRENT)>0L, all(PARAM$excluidas %in% CURRENT),
          !any(PARAM$metadata %in% CURRENT))
rm(muestra)
dataset <- generar_targets(dataset)       # el CSV crudo queda intacto
validar_target(dataset[foto_mes %in% 202103L:202106L])
validar_target(dataset[foto_mes==202108L],FALSE)

# Conteos publicados en la referencia V5.3; no se inventan ni se toman de caches privados.
conteos_pasquini <- data.table(
  foto_mes=c(rep(202103L:202106L,each=3),202107L,202107L,202108L),
  clase_ternaria=c(rep(c("BAJA+1","BAJA+2","CONTINUA"),4),NA_character_,"BAJA+1",NA_character_),
  N=c(1019L,960L,160921L,964L,1139L,161181L,1143L,870L,161755L,
      874L,1098L,162142L,163245L,1103L,164647L))
conteos <- dataset[,.N,by=.(foto_mes,clase_ternaria)]
setorderv(conteos,c("foto_mes","clase_ternaria"),na.last=TRUE)
setorderv(conteos_pasquini,c("foto_mes","clase_ternaria"),na.last=TRUE)
stopifnot(isTRUE(all.equal(as.data.frame(conteos),as.data.frame(conteos_pasquini),check.attributes=FALSE)))
UNIVERSO <- dataset[foto_mes==202108L,numero_de_cliente]
stopifnot(length(UNIVERSO)==164647L,!anyNA(UNIVERSO),!anyDuplicated(UNIVERSO))
DATA_ID <- firma(list(config=CONFIG_ID,raw_md5=raw_md5,CURRENT=CURRENT,
                     universo=as.character(UNIVERSO)))
if (is.null(abrir_checkpoint("datos/referencia",DATA_ID))) {
  d <- iniciar_checkpoint("datos/referencia")
  rds_atomico(list(url=url_oficial,raw_md5=raw_md5,data_id=DATA_ID),file.path(d,"fuente.rds"))
  csv_atomico(conteos,file.path(d,"conteos_mes_clase.csv"))
  csv_atomico(dataset[,.(numero_de_cliente,foto_mes,clase_ternaria,binaria2)],file.path(d,"targets.csv"))
  csv_atomico(data.table(numero_de_cliente=UNIVERSO),file.path(d,"universo_agosto.csv"))
  csv_atomico(fuentes,file.path(d,"fuentes_columnas.csv"))
  cerrar_checkpoint("datos/referencia",DATA_ID)
}

# Única corrección DQ. Se aplica en memoria ANTES de cualquier temporalización.
no_mayo <- dataset[foto_mes!=202105L,ccajas_depositos]
dataset[foto_mes==202105L,ccajas_depositos := NA_real_]
stopifnot(all(is.na(dataset[foto_mes==202105L,ccajas_depositos])),
          identical(no_mayo,dataset[foto_mes!=202105L,ccajas_depositos]))
rm(no_mayo)
print(conteos)


    foto_mes clase_ternaria      N
       <int>         <char>  <int>
 1:   202103         BAJA+1   1019
 2:   202103         BAJA+2    960
 3:   202103       CONTINUA 160921
 4:   202104         BAJA+1    964
 5:   202104         BAJA+2   1139
 6:   202104       CONTINUA 161181
 7:   202105         BAJA+1   1143
 8:   202105         BAJA+2    870
 9:   202105       CONTINUA 161755
10:   202106         BAJA+1    874
11:   202106         BAJA+2   1098
12:   202106       CONTINUA 162142
13:   202107         BAJA+1   1103
14:   202107           <NA> 163245
15:   202108           <NA> 164647


In [6]:
# BLOQUE 5 — RAW T2 → signed_rank mensual → exclusión → manifiesto congelado #####
prefijos <- c("","lag1_","delta1_","lag2_","delta2_")
bloques <- c("current","lag1","delta1","lag2","delta2")
RAW_T2 <- unlist(lapply(prefijos,function(p) paste0(p,CURRENT)),use.names=FALSE)
RANK_T2 <- paste0(RAW_T2,"_rank_signed")
stopifnot(!anyDuplicated(RANK_T2))

construir_mes <- function(origen,m,current=CURRENT) {
  actual <- origen[foto_mes==m]
  anterior1 <- origen[foto_mes==mes_anterior(m,1L)]
  anterior2 <- origen[foto_mes==mes_anterior(m,2L)]
  i1 <- match(actual$numero_de_cliente,anterior1$numero_de_cliente)
  i2 <- match(actual$numero_de_cliente,anterior2$numero_de_cliente)
  d <- actual[,.(numero_de_cliente,foto_mes,clase_ternaria,binaria2)]
  auditoria <- vector("list",length(current)*5L); k <- 0L
  # Se construyen las cinco versiones RAW de cada variable antes de rankearlas.
  # El orden final es por bloques, exactamente el de SAFE, no por variable.
  for (v in current) {
    x0 <- as.numeric(actual[[v]])
    x1 <- as.numeric(anterior1[[v]][i1])
    x2 <- as.numeric(anterior2[[v]][i2])
    raw <- list(x0,x1,x0-x1,x2,x0-x2)
    for (j in seq_along(raw)) {
      x <- raw[[j]]; z <- signed_rank(x)
      if (any(!is.na(x) & !is.finite(x)) || !identical(is.na(x),is.na(z)) ||
          any(z[!is.na(z)] < -1 | z[!is.na(z)] > 1) ||
          any(z[!is.na(x) & x==0]!=0) ||
          any(sign(z[!is.na(x)])!=sign(x[!is.na(x)])))
        stop("Rank/máscara/signos incorrectos en ",m," / ",v)
      nombre <- paste0(prefijos[j],v,"_rank_signed")
      set(d,j=nombre,value=z)
      k <- k+1L
      auditoria[[k]] <- data.table(foto_mes=m,feature=nombre,n_na=sum(is.na(x)),
        n_ceros=sum(x==0,na.rm=TRUE),sin_historia=if(j %in% c(2L,3L)) sum(is.na(i1)) else
          if(j %in% c(4L,5L)) sum(is.na(i2)) else 0L)
    }
  }
  nombres <- paste0(unlist(lapply(prefijos,function(p) paste0(p,current)),use.names=FALSE),"_rank_signed")
  setcolorder(d,c("numero_de_cliente","foto_mes","clase_ternaria","binaria2",nombres))
  validar_ids(d$numero_de_cliente,actual$numero_de_cliente)
  validar_target(d,m!=202108L)
  list(data=d,auditoria=rbindlist(auditoria))
}
for (m in PARAM$meses_cache) {
  key <- paste0("cache/mes_",m)
  sig <- firma(list(data=DATA_ID,mes=m,rank=RANK_T2))
  if (is.null(abrir_checkpoint(key,sig,remoto=FALSE))) {
    message("Construyendo ranks: ",m)
    d <- iniciar_checkpoint(key)
    z <- construir_mes(dataset,m)
    rds_atomico(z$data,file.path(d,"ranks.rds"))
    csv_atomico(z$auditoria,file.path(d,"auditoria_NA.csv"))
    # Caches regenerables: locales. Todos los resultados costosos van a GCS.
    cerrar_checkpoint(key,sig,remoto=FALSE)
    rm(z); gc(verbose=FALSE)
  }
}

# Los préstamos ya fueron temporalizados y rankeados sobre el mes COMPLETO.
# Ahora se excluyen únicamente del manifiesto que recibirá LightGBM.
manifiesto_total <- data.table(bloque=rep(bloques,each=length(CURRENT)),
  source_variable=rep(CURRENT,5L),source_feature=RAW_T2,model_feature=RANK_T2)
manifiesto_total[,incluida := !source_variable %in% PARAM$excluidas]
manifiesto <- manifiesto_total[incluida==TRUE]
manifiesto[,posicion := seq_len(.N)]
setcolorder(manifiesto,c("posicion","bloque","source_variable","source_feature","model_feature","incluida"))
FEATURES <- manifiesto$model_feature
FEATURE_ID <- firma(FEATURES)
stopifnot(length(FEATURES)==5L*(length(CURRENT)-2L),!anyDuplicated(FEATURES),
          all(grepl("_rank_signed$",FEATURES)),
          !any(grepl("(cprestamos_personales|mprestamos_personales)",FEATURES)))
PIPE_ID <- firma(list(data=DATA_ID,features=FEATURES))
if (is.null(abrir_checkpoint("features/manifiesto",PIPE_ID))) {
  d <- iniciar_checkpoint("features/manifiesto")
  csv_atomico(manifiesto,file.path(d,"features_finales.csv"))
  csv_atomico(manifiesto_total[incluida==FALSE],file.path(d,"prestamos_excluidos_RAW_y_RANK.csv"))
  csv_atomico(fuentes[decision!="RANK_PREDICTOR"],file.path(d,"otras_exclusiones.csv"))
  rds_atomico(list(FEATURES=FEATURES,feature_id=FEATURE_ID,pipe_id=PIPE_ID),file.path(d,"features.rds"))
  cerrar_checkpoint("features/manifiesto",PIPE_ID)
}
cargar_mes <- function(m) {
  key <- paste0("cache/mes_",m)
  sig <- firma(list(data=DATA_ID,mes=m,rank=RANK_T2))
  if (is.null(abrir_checkpoint(key,sig,remoto=FALSE))) stop("Falta cache: ",m)
  d <- readRDS(file.path(ck_dir(key),"ranks.rds"))
  stopifnot(identical(names(d)[-(1:4)],RANK_T2),all(d$foto_mes==m),
            !anyDuplicated(d$numero_de_cliente))
  validar_target(d,m!=202108L)
  if (m==202108L) validar_ids(d$numero_de_cliente,UNIVERSO)
  d[,c("numero_de_cliente","foto_mes","clase_ternaria","binaria2",FEATURES),with=FALSE]
}
cargar_meses <- function(meses) rbindlist(lapply(meses,cargar_mes),use.names=TRUE)
rm(dataset)
gc(verbose=FALSE)
message("Manifiesto congelado: ",length(FEATURES)," features; ",length(CURRENT),
        " fuentes rankeables antes de excluir préstamos.")
print(manifiesto[, .N,by=bloque])


Construyendo ranks: 202103

Construyendo ranks: 202104

Construyendo ranks: 202105

Construyendo ranks: 202106

Construyendo ranks: 202108



,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2330666,124.5,4511471,241.0,4511471,241.0
Vcells,4138477,31.6,277924520,2120.4,434254096,3313.1


Manifiesto congelado: 710 features; 144 fuentes rankeables antes de excluir préstamos.



    bloque     N
    <char> <int>
1: current   142
2:    lag1   142
3:  delta1   142
4:    lag2   142
5:  delta2   142


In [10]:
# BLOQUE 6 — BO SAFE: Kriging + EI; 10 LHS + 10 EGO #####
dominio_BO <- function() ps(
  num_iterations=p_int(lower=64L,upper=4096L),
  num_leaves=p_int(lower=16L,upper=1024L),
  min_sum_hessian_in_leaf=p_dbl(lower=1e-6,upper=0.1)
)
BO_ID <- firma(list(  pipe=PIPE_ID,  metodo="REGR_KM_BFGS_EI_RANDOM1000_EGO",  iniciales=6L,  EI=14L,  seed=727777L,  escala="LINEAL",  max_hojas=1024L,  version="BO_NUEVA"))
BO_CTX <- new.env(parent=emptyenv())

# Función global: al serializar el motor NO se incrustan matrices de entrenamiento.
objetivo_BO <- function(xs) {
  hp <- hp_lista(xs)
  if (!identical(hp,BO_CTX$hp)) stop("La propuesta BO no coincide con el checkpoint pendiente.")
  t0 <- proc.time()[["elapsed"]]
  resultado <- ajustar_SAFE(BO_CTX$X,BO_CTX$y,BO_CTX$Xvalid,hp,PARAM$seed_BO,
                            valid_y=BO_CTX$valid$binaria2)
  pred <- copy(BO_CTX$valid)
  pred[,prediction := resultado$p]
  auc <- auc_binaria(pred$binaria2,pred$prediction)
  curva <- curva_ganancia(pred,pred$prediction)
  stopifnot(max(PARAM$gainband)<=nrow(curva),is.finite(auc))
  BO_CTX$resultado <- list(pred=pred,curva=curva,meta=data.table(
    eval_id=BO_CTX$eval_id,origen=BO_CTX$origen,
    num_iterations=hp$num_iterations,num_leaves=hp$num_leaves,
    min_sum_hessian_in_leaf=hp$min_sum_hessian_in_leaf,
    seed=PARAM$seed_BO,AUC=auc,
    GainBand_junio=mean(curva$ganancia_acumulada[PARAM$gainband]),
    contexto="VALIDACION_TEMPORAL_SIN_HNC",feature_id=FEATURE_ID,
    n_train=nrow(BO_CTX$X),n_valid=nrow(pred),
    seg_entrenamiento=resultado$seg_entrenamiento,seg_prediccion=resultado$seg_prediccion,
    seg_evaluacion=proc.time()[["elapsed"]]-t0-resultado$seg_entrenamiento,
    seg_total=proc.time()[["elapsed"]]-t0))
  # Reset presente en SAFE efectivamente ejecutado y en Pasquini.
  set.seed(PARAM$seed_BO,kind="L'Ecuyer-CMRG",normal.kind="Inversion",sample.kind="Rejection")
  list(metrica=auc)
}
nuevo_motor_BO <- function(fun=objetivo_BO) {
  instancia <- OptimInstanceBatchSingleCrit$new(
    objective=ObjectiveRFun$new(fun=fun,domain=dominio_BO(),
      codomain=ps(metrica=p_dbl(tags="maximize")),id="BO_SAFE_FINAL"),
    terminator=trm("evals",n_evals=PARAM$n_total))
  surrogate <- srlrn(lrn("regr.km",optim.method="BFGS",nugget.estim=TRUE,
                         control=list(trace=FALSE)))
  acq <- acqf("ei")
  acqopt <- acqo(optimizer=opt("random_search",batch_size=1000),
                terminator=trm("evals",n_evals=1000))
  surrogate$archive <- instancia$archive
  acq$surrogate <- surrogate
  acqopt$acq_function <- acq
  list(instance=instancia,surrogate=surrogate,acq_function=acq,acq_optimizer=acqopt)
}
proponer_EI <- function(motor) {
  # Las mismas tres llamadas del cuerpo de bayesopt_ego(), separadas para poder
  # confirmar la propuesta ANTES de entrenar. No se cambia el optimizador de EI.
  motor$surrogate$archive <- motor$instance$archive
  motor$acq_function$surrogate <- motor$surrogate
  motor$acq_optimizer$acq_function <- motor$acq_function
  motor$surrogate$update()
  motor$acq_function$update()
  x <- motor$acq_optimizer$optimize()
  as.data.table(x)[,..hp_campos]
}
guardar_rng <- function() list(kind=RNGkind(),seed=get(".Random.seed",envir=.GlobalEnv))
restaurar_rng <- function(rng) {
  do.call(RNGkind,as.list(rng$kind))
  assign(".Random.seed",rng$seed,envir=.GlobalEnv)
}
validar_estado_BO <- function(z,n) {
  stopifnot(identical(z$bo_id,BO_ID),nrow(z$historia)==n,
            z$motor$instance$archive$n_evals==n,
            nrow(z$lhs)==6L,identical(names(z$lhs),hp_campos))
  if (n) {
    stopifnot(identical(as.integer(z$historia$eval_id),seq_len(n)))
    ar <- z$motor$instance$archive$data
    stopifnot(isTRUE(all.equal(ar$metrica,z$historia$AUC,tolerance=1e-14)))
    for (v in hp_campos) stopifnot(isTRUE(all.equal(as.numeric(ar[[v]]),
      as.numeric(z$historia[[v]]),tolerance=1e-14)))
  }
  invisible(TRUE)
}

ejecutar_BO <- function() {
  on.exit({rm(list=ls(BO_CTX),envir=BO_CTX);gc(verbose=FALSE)},add=TRUE)
  if (is.null(abrir_checkpoint("bo_n/inicio",BO_ID))) {
    set.seed(PARAM$seed_BO,kind="L'Ecuyer-CMRG",normal.kind="Inversion",sample.kind="Rejection")
    lhs <- paradox::generate_design_lhs(dominio_BO(),n=6L)$data
    lhs <- lhs[,..hp_campos]
    lhs[,`:=`(num_iterations=as.integer(num_iterations),num_leaves=as.integer(num_leaves))]
    stopifnot(nrow(lhs)==6L,!anyDuplicated(lhs))
    motor <- nuevo_motor_BO()
    set.seed(PARAM$seed_BO,kind="L'Ecuyer-CMRG",normal.kind="Inversion",sample.kind="Rejection")
    estado <- list(bo_id=BO_ID,motor=motor,lhs=lhs,historia=data.table(),
                   rng=guardar_rng(),seg_acumulados=0)
    d <- iniciar_checkpoint("bo_n/inicio")
    rds_atomico(estado,file.path(d,"estado.rds"))
    csv_atomico(lhs,file.path(d,"LHS_6.csv"))
    cerrar_checkpoint("bo_n/inicio",BO_ID)
  }
  ultima <- "bo_n/inicio"; n <- 0L
  # Recuperación automática de TODOS los trabajos confirmados, en orden; no replay de modelos.
  for (i in seq_len(PARAM$n_total)) {
    key <- sprintf("bo_n/eval_%02d",i)
    ck <- abrir_checkpoint(key,firma(list(bo=BO_ID,eval_id=i)))
    if (is.null(ck)) break
    ultima <- key; n <- i
  }
  estado <- readRDS(file.path(ck_dir(ultima),"estado.rds"))
  validar_estado_BO(estado,n)
  fijada <- abrir_checkpoint("bo_n/seleccion",BO_ID)
  if (!is.null(fijada) || solo_ensembles || !hay_tiempo() ||
      estado$seg_acumulados>=bo_horas*3600 || n==PARAM$n_total) return(estado)

  train <- cargar_meses(PARAM$BO_train); valid <- cargar_mes(PARAM$BO_valid)
  BO_CTX$X <- matriz_SAFE(train); BO_CTX$y <- train$binaria2
  BO_CTX$Xvalid <- matriz_SAFE(valid)
  BO_CTX$valid <- valid[,.(numero_de_cliente,foto_mes,clase_ternaria,binaria2)]
  rm(train,valid); gc(verbose=FALSE)
  base_tiempo <- estado$seg_acumulados
  inicio_sesion <- proc.time()[["elapsed"]]
  while (n<PARAM$n_total) {
    gastado <- base_tiempo+proc.time()[["elapsed"]]-inicio_sesion
    if (gastado>=bo_horas*3600 || !hay_tiempo()) break
    i <- n+1L
    key_p <- sprintf("bo_n/propuesta_%02d",i)
    sig_p <- firma(list(bo=BO_ID,propuesta=i))
    ck_p <- abrir_checkpoint(key_p,sig_p)
    if (!is.null(ck_p)) {
      pendiente <- readRDS(file.path(ck_dir(key_p),"pendiente.rds"))
      estado <- pendiente$estado
      hp <- pendiente$hp
      validar_estado_BO(estado,n)
      restaurar_rng(estado$rng)
      message("Reanudando la misma propuesta BO ",i)
    } else {
      restaurar_rng(estado$rng)
      hp <- if (i<=6L) copy(estado$lhs[i]) else proponer_EI(estado$motor)
      hp <- as.data.table(hp_lista(hp))
      # El continuo de Hessian hace improbable una repetición; nunca la reentrenamos silenciosamente.
      if (n && any(vapply(seq_len(n),function(k)
          identical(hp_lista(estado$historia[k]),hp_lista(hp)),logical(1))))
        stop("EI propuso HP ya evaluados; revisar la adquisición sin repetir entrenamiento.")
      estado$rng <- guardar_rng()
      d <- iniciar_checkpoint(key_p)
      rds_atomico(list(estado=estado,hp=hp),file.path(d,"pendiente.rds"))
      csv_atomico(hp,file.path(d,"hiperparametros.csv"))
      cerrar_checkpoint(key_p,sig_p)
    }
    # El backup de la propuesta también consume tiempo: volver a consultar el límite.
    if (base_tiempo+proc.time()[["elapsed"]]-inicio_sesion>=bo_horas*3600 || !hay_tiempo()) break
    BO_CTX$hp <- hp_lista(hp); BO_CTX$eval_id <- i
    BO_CTX$origen <- if (i<=6L) "LHS" else "EI"
    message("BO ",i,"/20: ",paste(unlist(BO_CTX$hp),collapse=" / "))
    restaurar_rng(estado$rng)
    estado$motor$instance$eval_batch(hp)
    z <- BO_CTX$resultado
    estado$historia <- rbindlist(list(estado$historia,z$meta),use.names=TRUE,fill=TRUE)
    estado$rng <- guardar_rng()
    estado$seg_acumulados <- base_tiempo+proc.time()[["elapsed"]]-inicio_sesion
    validar_estado_BO(estado,i)
    key <- sprintf("bo_n/eval_%02d",i); d <- iniciar_checkpoint(key)
    csv_atomico(z$pred,file.path(d,"predicciones_junio.csv"))
    csv_atomico(z$curva,file.path(d,"curva_junio_completa.csv"))
    csv_atomico(z$meta,file.path(d,"metricas.csv"))
    csv_atomico(estado$historia,file.path(d,"historial_BO.csv"))
    rds_atomico(estado,file.path(d,"estado.rds"))
    cerrar_checkpoint(key,firma(list(bo=BO_ID,eval_id=i)))
    n <- i
    message(sprintf("BO %d confirmada: AUC %.8f | GainBand %.3f M | %.1f min",
                    i,z$meta$AUC,z$meta$GainBand_junio/1e6,z$meta$seg_total/60))
    rm(z); BO_CTX$resultado <- NULL; gc(verbose=FALSE)
  }
  estado
}


In [11]:
respaldo_GCS <- FALSE
message("GCS interno desactivado; checkpoints locales activos")
print(respaldo_GCS)

GCS interno desactivado; checkpoints locales activos



[1] FALSE


In [12]:
print(dominio_BO())
print(c(
  total = PARAM$n_total,
  GCS_interno = respaldo_GCS,
  horas_BO = bo_horas
))

<ParamSet(3)>
                        id    class   lower  upper nlevels        default
                    <char>   <char>   <num>  <num>   <num>         <list>
1:          num_iterations ParamInt 6.4e+01 4096.0    4033 <NoDefault[0]>
2:              num_leaves ParamInt 1.6e+01 1024.0    1009 <NoDefault[0]>
3: min_sum_hessian_in_leaf ParamDbl 1.0e-06    0.1     Inf <NoDefault[0]>
    value
   <list>
1: [NULL]
2: [NULL]
3: [NULL]
      total GCS_interno    horas_BO 
         20           0           8 


In [17]:
PARAM$lgbm_fijos$num_threads <- 4L

stopifnot(param_modelo(
  list(
    num_iterations=50L,
    num_leaves=869L,
    min_sum_hessian_in_leaf=0.096488455638161
  ),
  727777L
)$num_threads == 4L)

cat("SAFE configurado con 4 hilos\n")

SAFE configurado con 4 hilos


In [18]:
# BLOQUE 7 — Ejecutar/reanudar BO y congelar un único HP ganador #####
estado_BO <- ejecutar_BO()
HIST_BO <- estado_BO$historia
if (!nrow(HIST_BO)) stop("No hay evaluaciones BO terminadas. Ampliar bo_horas/límite y reanudar.")
if (is.null(abrir_checkpoint("bo_n/seleccion",BO_ID))) {
  if (solo_ensembles) stop("No existe selección congelada; ejecutar el pipeline para crearla.")
  ranking <- copy(HIST_BO)
  setorder(ranking,-AUC,eval_id)
  seleccion <- list(hp=hp_lista(ranking[1]),eval_id=ranking$eval_id[1],
    auc=ranking$AUC[1],gainband=ranking$GainBand_junio[1],n_evaluaciones=nrow(HIST_BO),
    n_LHS=sum(HIST_BO$origen=="LHS"),n_EI=sum(HIST_BO$origen=="EI"),
    bo_id=BO_ID,feature_id=FEATURE_ID,criterio="AUC_JUNIO_SIN_HNC",
    cierre=if(nrow(HIST_BO)==20L) "20_EVALUACIONES" else "PRESUPUESTO_O_LIMITE")
  d <- iniciar_checkpoint("bo_n/seleccion")
  rds_atomico(seleccion,file.path(d,"seleccion.rds"))
  csv_atomico(ranking,file.path(d,"ranking_AUC.csv"))
  cerrar_checkpoint("bo_n/seleccion",BO_ID)
}
seleccion <- readRDS(file.path(ck_dir("bo_n/seleccion"),"seleccion.rds"))
HP <- seleccion$hp
HP_ID <- firma(list(bo=BO_ID,seleccion=seleccion))
stopifnot(seleccion$n_evaluaciones==nrow(HIST_BO),identical(seleccion$feature_id,FEATURE_ID))
message("HP congelados tras ",seleccion$n_evaluaciones," evaluaciones (",
        seleccion$n_LHS," LHS + ",seleccion$n_EI," EI). BO posterior no se mezcla con esta producción.")
print(data.table(eval_id=seleccion$eval_id,AUC=seleccion$auc,
  GainBand_junio_sin_HNC=seleccion$gainband,as.data.table(HP)))
rm(estado_BO)
gc(verbose=FALSE)


Reanudando la misma propuesta BO 1

BO 1/20: 3266 / 869 / 0.096488455638161



INFO  [14:04:52.950] [bbotk] Evaluating 1 configuration(s)
INFO  [14:17:32.145] [bbotk] Result of batch 1:
INFO  [14:17:32.155] [bbotk]  num_iterations num_leaves min_sum_hessian_in_leaf   metrica
INFO  [14:17:32.155] [bbotk]            3266        869              0.09648846 0.9198246


BO 1 confirmada: AUC 0.91982464 | GainBand 444.125 M | 12.7 min

BO 2/20: 1381 / 657 / 0.018217157942036



INFO  [14:17:32.751] [bbotk] Evaluating 1 configuration(s)
INFO  [14:22:19.185] [bbotk] Result of batch 2:
INFO  [14:22:19.188] [bbotk]  num_iterations num_leaves min_sum_hessian_in_leaf   metrica
INFO  [14:22:19.188] [bbotk]            1381        657              0.01821716 0.9185337


BO 2 confirmada: AUC 0.91853373 | GainBand 448.250 M | 4.8 min

BO 3/20: 2426 / 135 / 0.00482339022717023



...

BO 19 confirmada: AUC 0.91987785 | GainBand 447.288 M | 9.0 min

BO 20/20: 3938 / 201 / 0.0473186024263462



INFO  [17:03:12.148] [bbotk] Evaluating 1 configuration(s)
INFO  [17:12:21.002] [bbotk] Result of batch 20:
INFO  [17:12:21.006] [bbotk]  num_iterations num_leaves min_sum_hessian_in_leaf   metrica
INFO  [17:12:21.006] [bbotk]            3938        201               0.0473186 0.9161953


BO 20 confirmada: AUC 0.91619525 | GainBand 435.050 M | 9.1 min

HP congelados tras 20 evaluaciones (6 LHS + 14 EI). BO posterior no se mezcla con esta producción.



   eval_id       AUC GainBand_junio_sin_HNC num_iterations num_leaves
     <int>     <num>                  <num>          <int>      <int>
1:       9 0.9199935              446462500           3246        951
   min_sum_hessian_in_leaf
                     <num>
1:             0.009123316


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2649115,141.5,4511471,241.0,4511471,241.0
Vcells,469076445,3578.8,1540067390,11749.8,1540020742,11749.5


In [19]:
# BLOQUE 8 — HNC original: cinco folds por cliente, una semilla, 5% por mes #####
validar_claves <- function(a,b) {
  if (nrow(a)!=nrow(b) || anyDuplicated(a,by=c("numero_de_cliente","foto_mes")) ||
      !identical(as.character(a$numero_de_cliente),as.character(b$numero_de_cliente)) ||
      !identical(as.integer(a$foto_mes),as.integer(b$foto_mes)))
    stop("OOF: claves cliente-mes desalineadas/duplicadas.")
  invisible(TRUE)
}
make_folds <- function(dt) {
  # El HNC original es autónomo y usa set.seed(seed+y), sin cambiar RNGkind.
  # Fijamos explícitamente el default de una sesión R nueva, para que la BO previa
  # (L'Ecuyer) no altere accidentalmente los folds del notebook HNC original.
  anterior <- guardar_rng(); on.exit(restaurar_rng(anterior),add=TRUE)
  RNGkind("Mersenne-Twister","Inversion","Rejection")
  grp <- dt[,.(ymax=max(binaria2)),by=numero_de_cliente]
  grp[,fold := NA_integer_]
  for (y in c(0L,1L)) {
    idx <- which(grp$ymax==y)
    set.seed(PARAM$seed_OOF+y)
    if (length(idx)>1L) idx <- sample(idx,length(idx))
    grp$fold[idx] <- rep(seq_len(PARAM$n_folds),length.out=length(idx))
  }
  folds <- grp$fold[match(dt$numero_de_cliente,grp$numero_de_cliente)]
  stopifnot(!anyNA(folds),!anyDuplicated(grp$numero_de_cliente),
            all(folds %in% seq_len(PARAM$n_folds)))
  list(grp=grp,fold=folds)
}
marcar_HNC <- function(dt,prob,folds) {
  validar_prob(prob,nrow(dt))
  stopifnot(length(folds)==nrow(dt),!anyNA(folds))
  out <- dt[,.(numero_de_cliente,foto_mes,clase_ternaria,binaria2)]
  out[,`:=`(prob_OOF=prob,fold=as.integer(folds),excluir=FALSE)]
  for (m in sort(unique(out$foto_mes))) {
    idx <- which(out$foto_mes==m & out$clase_ternaria=="CONTINUA")
    k <- as.integer(floor(length(idx)*PARAM$hnc_pct/100))
    if (k) {
      ix <- idx[order(-prob[idx],as.character(out$numero_de_cliente[idx]))[seq_len(k)]]
      set(out,i=ix,j="excluir",value=TRUE)
    }
  }
  resumen <- out[,.(continua=sum(clase_ternaria=="CONTINUA"),
    eliminados=sum(excluir),bajas_antes=sum(binaria2==1L),
    bajas_despues=sum(binaria2==1L & !excluir),bajas_eliminadas=sum(binaria2==1L & excluir)),by=foto_mes]
  stopifnot(all(resumen$eliminados==floor(resumen$continua*PARAM$hnc_pct/100)),
            all(resumen$bajas_eliminadas==0L),all(out$clase_ternaria[out$excluir]=="CONTINUA"))
  list(manifiesto=out,resumen=resumen)
}
HNC_BASE_ID <- firma(list(pipe=PIPE_ID,hp=HP_ID,meses=c(202105L,202106L),
  seed=727777L,folds=5L,pct=5,RNG="Mersenne-Twister/Inversion/Rejection",
  desempate="order(-prob_OOF,as.character(numero_de_cliente))"))

ejecutar_HNC <- function() {
  ck <- abrir_checkpoint("hnc/limpieza",HNC_BASE_ID)
  if (!is.null(ck)) return(readRDS(file.path(ck_dir("hnc/limpieza"),"limpieza.rds")))
  if (solo_ensembles) stop("Aún no existe limpieza HNC confirmada.")
  dt <- cargar_meses(c(202105L,202106L))
  ff <- make_folds(dt)
  fold_id <- firma(as.data.frame(ff$grp))
  sig_f <- firma(list(hnc=HNC_BASE_ID,folds=fold_id))
  if (is.null(abrir_checkpoint("hnc/folds",sig_f))) {
    d <- iniciar_checkpoint("hnc/folds")
    rds_atomico(ff,file.path(d,"folds.rds"))
    csv_atomico(ff$grp,file.path(d,"folds_por_cliente.csv"))
    cerrar_checkpoint("hnc/folds",sig_f)
  } else {
    guardado <- readRDS(file.path(ck_dir("hnc/folds"),"folds.rds"))
    stopifnot(identical(guardado$fold,ff$fold),
              identical(as.data.frame(guardado$grp),as.data.frame(ff$grp)))
  }
  pred <- rep(NA_real_,nrow(dt)); X <- NULL
  for (f in seq_len(PARAM$n_folds)) {
    it <- which(ff$fold!=f); iv <- which(ff$fold==f)
    stopifnot(length(it)>0L,length(iv)>0L,
      !length(intersect(unique(dt$numero_de_cliente[it]),unique(dt$numero_de_cliente[iv]))))
    key <- sprintf("hnc/fold_%02d",f)
    sig <- firma(list(hnc=HNC_BASE_ID,fold_id=fold_id,fold=f))
    ck <- abrir_checkpoint(key,sig)
    if (is.null(ck)) {
      if (!hay_tiempo()) {
        message("Límite alcanzado: HNC pendiente desde fold ",f,"; folds previos conservados.")
        return(NULL)
      }
      if (is.null(X)) X <- matriz_SAFE(dt)
      d <- iniciar_checkpoint(key)
      message("OOF HNC fold ",f,"/5; seed 727777; HP BO ",seleccion$eval_id)
      z <- ajustar_SAFE(X[it,,drop=FALSE],dt$binaria2[it],X[iv,,drop=FALSE],HP,PARAM$seed_OOF,
        modelo_archivo=if(guardar_modelos) file.path(d,"modelo.txt") else NULL)
      p <- dt[iv,.(numero_de_cliente,foto_mes,clase_ternaria,binaria2)]
      p[,`:=`(fold=f,prob_OOF=z$p)]
      csv_atomico(p,file.path(d,"predicciones_OOF.csv"))
      rds_atomico(list(hp=HP,hp_id=HP_ID,feature_id=FEATURE_ID,seed=PARAM$seed_OOF,
        fold=f,fold_id=fold_id,n_train=length(it),n_valid=length(iv),
        sin_clientes_compartidos=TRUE,seg_entrenamiento=z$seg_entrenamiento,
        seg_prediccion=z$seg_prediccion),file.path(d,"meta.rds"))
      cerrar_checkpoint(key,sig)
      rm(z,p); gc(verbose=FALSE)
    }
    p <- fread(file.path(ck_dir(key),"predicciones_OOF.csv"),na.strings=c("","NA"))
    validar_claves(p,dt[iv]); validar_prob(p$prob_OOF,length(iv))
    stopifnot(all(p$fold==f))
    pred[iv] <- p$prob_OOF
  }
  validar_prob(pred,nrow(dt))
  clean <- marcar_HNC(dt,pred,ff$fold)
  clean$clean_id <- firma(list(hnc=HNC_BASE_ID,fold_id=fold_id,
    ids=as.character(clean$manifiesto$numero_de_cliente),mes=clean$manifiesto$foto_mes,
    prob=clean$manifiesto$prob_OOF,excluir=clean$manifiesto$excluir))
  clean$hp_id <- HP_ID; clean$feature_id <- FEATURE_ID
  clean$version <- paste0("HNC5_",substr(clean$clean_id,1L,12L))
  d <- iniciar_checkpoint("hnc/limpieza")
  csv_atomico(clean$manifiesto,file.path(d,"manifiesto_HNC_mayo_junio.csv"))
  csv_atomico(clean$resumen,file.path(d,"conteos_HNC_por_mes.csv"))
  rds_atomico(clean,file.path(d,"limpieza.rds"))
  cerrar_checkpoint("hnc/limpieza",HNC_BASE_ID)
  clean
}


In [20]:
# BLOQUE 9 — Generar/reutilizar la única limpieza de mayo + junio #####
HNC <- ejecutar_HNC()
if (!is.null(HNC)) {
  stopifnot(identical(HNC$hp_id,HP_ID),identical(HNC$feature_id,FEATURE_ID),
            all(HNC$resumen$bajas_eliminadas==0L))
  print(HNC$resumen)
  message("Limpieza confirmada: ",HNC$version)
}
gc(verbose=FALSE)


OOF HNC fold 1/5; seed 727777; HP BO 9

OOF HNC fold 2/5; seed 727777; HP BO 9

OOF HNC fold 3/5; seed 727777; HP BO 9

OOF HNC fold 4/5; seed 727777; HP BO 9

OOF HNC fold 5/5; seed 727777; HP BO 9



   foto_mes continua eliminados bajas_antes bajas_despues bajas_eliminadas
      <int>    <int>      <int>       <int>         <int>            <int>
1:   202105   161755       8087        2013          2013                0
2:   202106   162142       8107        1972          1972                0


Limpieza confirmada: HNC5_6b96eb33a574



,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2652141,141.7,4511471,241.0,4511471,241.0
Vcells,470724032,3591.4,1540067390,11749.8,1540020742,11749.5


In [21]:
# BLOQUE 10 — Producción progresiva y ensembles por promedio de probabilidades #####
clave_semilla <- function(P,s) paste0("produccion/",P,"/seed_",s)
firma_semilla <- function(P,s) firma(list(pipe=PIPE_ID,hp=HP_ID,hnc=HNC$clean_id,
                                        P=P,seed=as.integer(s),train=PARAM$P_final[[P]]))
etiqueta <- function(P) paste0(P,"_HP_BO",sprintf("%02d",seleccion$eval_id),"_",
                               substr(HP_ID,1,8),"_",HNC$version)
leer_pred_semilla <- function(P,s) {
  key <- clave_semilla(P,s)
  if (is.null(abrir_checkpoint(key,firma_semilla(P,s)))) return(NULL)
  meta <- readRDS(file.path(ck_dir(key),"meta.rds"))
  stopifnot(identical(meta$hp_id,HP_ID),identical(meta$feature_id,FEATURE_ID),
            identical(meta$clean_id,HNC$clean_id),identical(meta$P,P),meta$seed==s)
  p <- fread(file.path(ck_dir(key),meta$pred_file),na.strings=c("","NA"))
  validar_ids(p$numero_de_cliente,UNIVERSO); validar_prob(p$prediction,length(UNIVERSO))
  p
}
generar_ensemble <- function(P) {
  semillas <- PARAM$semillas[seq_len(n_semillas)]
  suma <- rep(0,length(UNIVERSO)); usadas <- integer()
  for (s in semillas) {
    p <- leer_pred_semilla(P,s)
    if (is.null(p)) next
    suma <- suma+p$prediction
    usadas <- c(usadas,s)
  }
  n <- length(usadas)
  if (!n) return(list(P=P,n=0L,completo=FALSE,carpeta=NA_character_))
  sig <- firma(list(pipe=PIPE_ID,hp=HP_ID,hnc=HNC$clean_id,P=P,semillas=usadas,
                    metodo="MEDIA_ARITMETICA_PROBABILIDADES",cortes=PARAM$cortes))
  key <- paste0("ensembles/",P,"/n",sprintf("%02d",n),"_",substr(sig,1,12))
  ck <- abrir_checkpoint(key,sig)
  if (is.null(ck)) {
    d <- iniciar_checkpoint(key)
    nombre <- paste0(etiqueta(P),"_n",sprintf("%02d",n))
    ens <- data.table(numero_de_cliente=UNIVERSO,prediction=suma/n)
    validar_prob(ens$prediction,length(UNIVERSO)); validar_ids(ens$numero_de_cliente,UNIVERSO)
    csv_atomico(ens,file.path(d,paste0(nombre,"_probabilidades_agosto.csv")))
    setorder(ens,-prediction,numero_de_cliente)
    ens[,ranking := seq_len(.N)]
    csv_atomico(ens,file.path(d,paste0(nombre,"_ranking_agosto.csv")))
    salidas <- vector("list",length(PARAM$cortes))
    for (j in seq_along(PARAM$cortes)) {
      k <- PARAM$cortes[j]
      stopifnot(k<=nrow(ens))
      nombre_sub <- paste0(nombre,"_top",k,".csv")
      f <- file.path(d,nombre_sub)
      csv_atomico(ens[seq_len(k),.(numero_de_cliente)],f,encabezado=FALSE)
      ids <- fread(f,header=FALSE)[[1L]]
      validar_ids(ids,ens$numero_de_cliente[seq_len(k)])
      stopifnot(length(ids)==k,all(ids %in% UNIVERSO))
      salidas[[j]] <- data.table(modelo=P,eval_BO=seleccion$eval_id,
        num_iterations=HP$num_iterations,num_leaves=HP$num_leaves,
        min_sum_hessian_in_leaf=HP$min_sum_hessian_in_leaf,
        n_semillas=n,cutoff=k,HNC=HNC$version,archivo=nombre_sub,md5=md5(f))
    }
    csv_atomico(rbindlist(salidas),file.path(d,"manifest_submissions.csv"))
    rds_atomico(list(P=P,hp=HP,hp_id=HP_ID,feature_id=FEATURE_ID,clean_id=HNC$clean_id,
      HNC_version=HNC$version,semillas=usadas,n_semillas=n,
      objetivo_al_crear=n_semillas,provisional_al_crear=n<n_semillas,
      agregacion="promedio aritmetico de probabilidades",desempate="ID numerico ascendente"),
      file.path(d,"meta.rds"))
    cerrar_checkpoint(key,sig)
    message(P,": ensemble de ",n," semillas y ",length(PARAM$cortes)," submissions confirmados.")
  }
  list(P=P,n=n,completo=n==n_semillas,carpeta=ck_dir(key),checkpoint=key)
}
entrenamiento_final <- function(P) {
  dt <- cargar_meses(PARAM$P_final[[P]])
  claves <- paste(HNC$manifiesto$foto_mes,HNC$manifiesto$numero_de_cliente,sep="|")
  idx <- match(paste(dt$foto_mes,dt$numero_de_cliente,sep="|"),claves)
  mayo_junio <- dt$foto_mes %in% c(202105L,202106L)
  stopifnot(!anyNA(idx[mayo_junio]),all(is.na(idx[!mayo_junio])))
  quitar <- rep(FALSE,nrow(dt))
  quitar[mayo_junio] <- HNC$manifiesto$excluir[idx[mayo_junio]]
  stopifnot(!any(dt$binaria2[quitar]==1L),
    sum(quitar)==sum(HNC$resumen$eliminados),
    !any(quitar[dt$foto_mes %in% c(202103L,202104L)]))
  dt[!quitar]                             # NUNCA recalcular ranks después de HNC
}
ejecutar_produccion <- function(P) {
  # Primero recupera/exporta todo lo disponible, incluso si ya venció el límite.
  estado <- generar_ensemble(P)
  if (estado$completo || solo_ensembles || !hay_tiempo()) return(estado)
  train <- entrenamiento_final(P)
  score <- cargar_mes(PARAM$mes_score)
  validar_ids(score$numero_de_cliente,UNIVERSO)
  X <- matriz_SAFE(train); y <- train$binaria2; Xscore <- matriz_SAFE(score)
  conteos_train <- train[,.N,by=.(foto_mes,clase_ternaria)]
  n_train <- nrow(train)
  rm(train,score); gc(verbose=FALSE)
  for (s in PARAM$semillas[seq_len(n_semillas)]) {
    key <- clave_semilla(P,s); sig <- firma_semilla(P,s)
    if (!is.null(abrir_checkpoint(key,sig))) next
    if (!hay_tiempo()) break
    d <- iniciar_checkpoint(key)
    message(P," seed ",s,": train ",n_train," / agosto ",length(UNIVERSO))
    z <- ajustar_SAFE(X,y,Xscore,HP,s,
      modelo_archivo=if(guardar_modelos) file.path(d,"modelo.txt") else NULL)
    nombre <- paste0(etiqueta(P),"_seed",s,"_agosto.csv")
    p <- data.table(numero_de_cliente=UNIVERSO,prediction=z$p)
    validar_ids(p$numero_de_cliente,UNIVERSO)
    csv_atomico(p,file.path(d,nombre))
    csv_atomico(conteos_train,file.path(d,"filas_train.csv"))
    rds_atomico(list(P=P,seed=as.integer(s),hp=HP,hp_id=HP_ID,feature_id=FEATURE_ID,
      clean_id=HNC$clean_id,HNC_version=HNC$version,fijos=PARAM$lgbm_fijos,
      train=PARAM$P_final[[P]],score=PARAM$mes_score,n_train=n_train,n_score=length(UNIVERSO),
      pred_file=nombre,seg_entrenamiento=z$seg_entrenamiento,seg_prediccion=z$seg_prediccion),
      file.path(d,"meta.rds"))
    cerrar_checkpoint(key,sig)             # antes de la siguiente semilla
    rm(z,p); gc(verbose=FALSE)
    estado <- generar_ensemble(P)          # siempre queda una submission parcial utilizable
  }
  generar_ensemble(P)
}


In [22]:
# BLOQUE 11 — P2 completo y respaldado → P3; sin mezclar familias #####
resultados <- list()
if (is.null(HNC)) {
  message("HNC aún incompleta; reanudar este notebook. No se inició producción.")
} else {
  if (producir_P2) resultados$P2 <- ejecutar_produccion("P2")
  if (producir_P3) {
    if (!producir_P2 || isTRUE(resultados$P2$completo)) {
      resultados$P3 <- ejecutar_produccion("P3")
    } else {
      # P3 ya existente puede exportarse con solo_ensembles, sin iniciar ningún modelo.
      if (solo_ensembles) resultados$P3 <- generar_ensemble("P3")
      message("P3 pendiente: completar y respaldar P2, o desactivar producir_P2 explícitamente.")
    }
  }
}
if (length(resultados)) {
  resumen <- rbindlist(lapply(resultados,function(z) data.table(
    modelo=z$P,semillas_completas=z$n,objetivo=n_semillas,
    estado=if(z$completo) "COMPLETO" else "PROVISIONAL",carpeta=z$carpeta)))
  print(resumen)
  resumen_id <- firma(as.data.frame(resumen))
  key <- paste0("reportes/resumen_",substr(resumen_id,1,16))
  if (is.null(abrir_checkpoint(key,resumen_id))) {
    d <- iniciar_checkpoint(key)
    csv_atomico(resumen,file.path(d,"salidas_disponibles.csv"))
    cerrar_checkpoint(key,resumen_id)
  }
}
message("BO: ",nrow(HIST_BO),"/20 evaluaciones; AUC elegida=",round(seleccion$auc,8),
  "; GainBand junio SIN HNC=",round(seleccion$gainband/1e6,3)," M.")
message("Submissions: una columna de IDs, sin encabezado. No se envían automáticamente.")


P2 seed 727777: train 311688 / agosto 164647

P2: ensemble de 1 semillas y 12 submissions confirmados.

P2 seed 773777: train 311688 / agosto 164647

P2: ensemble de 2 semillas y 12 submissions confirmados.

P2 seed 775777: train 311688 / agosto 164647

P2: ensemble de 3 semillas y 12 submissions confirmados.

P2 seed 777737: train 311688 / agosto 164647

P2: ensemble de 4 semillas y 12 submissions confirmados.

P2 seed 787777: train 311688 / agosto 164647

P2: ensemble de 5 semillas y 12 submissions confirmados.

P2 seed 777727: train 311688 / agosto 164647

P2: ensemble de 6 semillas y 12 submissions confirmados.

P2 seed 777773: train 311688 / agosto 164647

P2: ensemble de 7 semillas y 12 submissions confirmados.

P2 seed 777775: train 311688 / agosto 164647

P2: ensemble de 8 semillas y 12 submissions confirmados.

P2 seed 777787: train 311688 / agosto 164647

P2: ensemble de 9 semillas y 12 submissions confirmados.

P2 seed 797777: train 311688 / agosto 164647

P2: ensemble de 10

## Validación realizada antes de entregar

Se ejecutaron pruebas pequeñas en **R 4.3.3**, con `mlr3mbo 1.2.1` y `bbotk 1.13.1`, sobre las funciones de este notebook:

| Prueba | Resultado |
|---|---|
| Parseo de todas las celdas de código R; estructura JSON y kernel R | Correcto. Celdas sin resultados de entrenamientos precargados. |
| Targets: BAJA+1, BAJA+2, desapariciones, huecos de meses y agosto desconocido | Correcto en casos pequeños con respuestas conocidas. |
| Signed rank: negativos, positivos, empates, cero y NA | Coincide con resultados esperados; no imputa valores. |
| Temporales por ID/mes: cliente sin abril; mayo corregido propagado a junio; marzo sin historia | Correcto. No desplaza la observación de otro mes para rellenar el faltante. |
| Folds HNC | Coinciden con la función original bajo el RNG de una sesión autónoma. Sin clientes compartidos entre train/valid del fold. |
| HNC 5% y empates | Cantidad exacta por `floor`, desempate por ID como texto y ninguna BAJA eliminada. |
| Checkpoint, firma incorrecta y archivo corrupto | Checkpoint válido aceptado; los otros dos rechazados. |
| GCS simulado: fallo de subida, reintento y pérdida de archivos locales | Se bloquea el avance; se completa el respaldo pendiente y se restauran bytes verificados. |
| Bucle EGO: función matemática pequeña | Las 10 propuestas EI coinciden con `bayesopt_ego()` nativo tras el mismo LHS, incluso serializando/restaurando el motor entre pasos. |
| Interrupción BO en la evaluación 17 y fallo después del commit local de la 18 | Reanuda la misma secuencia; sólo se repite el intento interrumpido de la 17. Ninguna evaluación completada vuelve a entrenarse. |
| BO parcial y selección congelada | Selecciona con dos evaluaciones disponibles; ampliar posteriormente el presupuesto no cambia una selección ya cerrada. |
| Flujo HNC → P2 → P3 y ampliación P2 de 10 a 20 semillas | Una HNC reutilizada; sólo diez semillas adicionales al ampliar; archivos anteriores intactos. |
| Ensemble y submissions | Promedio aritmético de probabilidades, ranking consistente, IDs exactos y exportación parcial con el número real de semillas. |

**Límites de esta validación:** los ensayos de orquestación usaron un sustituto determinista del entrenamiento LightGBM sobre datos diminutos; la BO de prueba sí utilizó el surrogate y la adquisición reales de `mlr3mbo`. No se entrenaron modelos de la competencia ni se descargó el dataset completo para esta auditoría. Tampoco se accedió al bucket privado: el protocolo GCS se probó con un CLI simulado y sus llamadas se contrastaron con la documentación oficial. La primera ejecución real verificará permisos, versiones, conteos, matrices y transferencias en la VM.

Las pruebas pequeñas integradas del bloque 3b se ejecutan al abrir el pipeline; las comprobaciones de datos, OOF y archivos están incorporadas a sus etapas. No se adjunta ningún script auxiliar.
